# Pipe CFD Post-Processing Workflow

In [1]:
from pathlib import Path
import csv, math, re, statistics

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
CASE_DIR = ROOT / "openfoam" / "pipe"
OUT_DIR = ROOT / "output" / "pipe" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RHO = 1060.0      # kg/m3 --> (conversion from incompressible kinematic pressure to Pa
MU = 3.5e-3       # Pa s
RADIUS = 0.002    # m
LENGTH = 0.040    # m
AREA = math.pi * RADIUS**2

print(f"Case directory: {CASE_DIR}")
print(f"Output directory: {OUT_DIR}")

Case directory: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/openfoam/pipe
Output directory: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/output/pipe/analysis


## Step 1: CFD Outputs

In [2]:
def latest_time_dir(parent):
    if not parent.exists():
        return None
    dirs = [p for p in parent.iterdir() if p.is_dir()]
    if not dirs:
        return None
    return max(dirs, key=lambda p: float(p.name) if p.name.replace(".", "", 1).isdigit() else -1)

def discover_outputs(case_dir):
    sample_time = latest_time_dir(case_dir / "postProcessing" / "sampleDict")
    wss_time = latest_time_dir(case_dir / "postProcessing" / "wallShearStress")
    items = [
        ("postProcessing/", case_dir / "postProcessing", "OpenFOAM post-processing directory."),
        ("centreline.xy", sample_time / "centreline.xy" if sample_time else case_dir / "postProcessing/sampleDict/<time>/centreline.xy", "Centreline pressure/velocity sample."),
        ("outletRadial.xy", sample_time / "outletRadial.xy" if sample_time else case_dir / "postProcessing/sampleDict/<time>/outletRadial.xy", "Outlet radial velocity profile sample."),
        ("inlet areaAverage(U)", case_dir / "postProcessing/patchAverage(patch=inlet,fields=(U))/144/surfaceFieldValue.dat", "Inlet mean velocity for Q."),
        ("outlet areaAverage(U)", case_dir / "postProcessing/patchAverage(patch=outlet,fields=(U))/144/surfaceFieldValue.dat", "Outlet mean velocity for Q."),
        ("inlet areaAverage(p)", case_dir / "postProcessing/patchAverage(patch=inlet,fields=(p))/144/surfaceFieldValue.dat", "Inlet patch pressure."),
        ("outlet areaAverage(p)", case_dir / "postProcessing/patchAverage(patch=outlet,fields=(p))/144/surfaceFieldValue.dat", "Outlet patch pressure."),
        ("wallShearStress.dat", wss_time / "wallShearStress.dat" if wss_time else case_dir / "postProcessing/wallShearStress/<time>/wallShearStress.dat", "Wall shear stress diagnostic."),
        ("ParaView state/export", case_dir / "postProcessing/pipe_postprocessing.pvsm", "Saved visualization state/export."),
    ]
    rows = [{"File": n, "Exists": p.exists(), "Purpose": purpose, "Path": str(p)} for n, p, purpose in items]
    return rows

discovery = discover_outputs(CASE_DIR)
for row in discovery:
    print(f"{row['File']:<24} exists={str(row['Exists']):<5} {row['Purpose']}")

print("\nIf files are missing, generate them with commands such as:")
print("  postProcess -func sampleDict -latestTime")
print("  postProcess -func 'patchAverage(patch=inlet,fields=(U))' -latestTime")
print("  postProcess -func 'patchAverage(patch=outlet,fields=(U))' -latestTime")
print("  postProcess -func 'patchAverage(patch=inlet,fields=(p))' -latestTime")
print("  postProcess -func 'patchAverage(patch=outlet,fields=(p))' -latestTime")
print("  simpleFoam -postProcess -func wallShearStress -latestTime")

postProcessing/          exists=True  OpenFOAM post-processing directory.
centreline.xy            exists=True  Centreline pressure/velocity sample.
outletRadial.xy          exists=True  Outlet radial velocity profile sample.
inlet areaAverage(U)     exists=True  Inlet mean velocity for Q.
outlet areaAverage(U)    exists=True  Outlet mean velocity for Q.
inlet areaAverage(p)     exists=True  Inlet patch pressure.
outlet areaAverage(p)    exists=True  Outlet patch pressure.
wallShearStress.dat      exists=True  Wall shear stress diagnostic.
ParaView state/export    exists=True  Saved visualization state/export.

If files are missing, generate them with commands such as:
  postProcess -func sampleDict -latestTime
  postProcess -func 'patchAverage(patch=inlet,fields=(U))' -latestTime
  postProcess -func 'patchAverage(patch=outlet,fields=(U))' -latestTime
  postProcess -func 'patchAverage(patch=inlet,fields=(p))' -latestTime
  postProcess -func 'patchAverage(patch=outlet,fields=(p))' -late

## Step 2: Load and Inspect Data

`centreline.xy` : values sampled along the pipe axis used for pressure-gradient.

`outletRadial.xy` : a diameter/radius sample at the outlet,  used to compare the CFD velocity profile with the analytical Poiseuille profile.

In [4]:
def read_table_preview(path, n=5):
    with open(path, newline='') as f:
        rows = list(csv.DictReader(f))
    return rows[:n], list(rows[0].keys()) if rows else []

for csv_name in ["centreline_loaded.csv", "outlet_radial_loaded.csv", "flow_rate_summary.csv"]:
    path = OUT_DIR / csv_name
    if path.exists():
        preview, columns = read_table_preview(path)
        print(f"\n{csv_name}: {len(list(csv.DictReader(open(path))))} samples")
        print("columns:", columns)
        for row in preview:
            print(row)
    else:
        print(f"Missing {csv_name}; dependent sections will be skipped.")

print("\nUnits: coordinates & distance [m] , velocity [m/s], OpenFOAM incompressible p is kinematic pressure [m2/s2], converted to Pa with rho=1060 kg/m3.")


centreline_loaded.csv: 200 samples
columns: ['distance', 'x', 'y', 'z', 'U_x', 'U_y', 'U_z', 'p']
{'distance': '1.5700925e-20', 'x': '1.110223e-20', 'y': '1.110223e-20', 'z': '0.001', 'U_x': '2.8125e-12', 'U_y': '3.675e-12', 'U_z': '0.19872576', 'p': '0.026363281'}
{'distance': '0.00019095477', 'x': '1.110223e-20', 'y': '1.110223e-20', 'z': '0.0011909548', 'U_x': '4.0751884e-12', 'U_y': '5.1405779e-12', 'U_z': '0.19880656', 'p': '0.026198824'}
{'distance': '0.00038190955', 'x': '1.110223e-20', 'y': '1.110223e-20', 'z': '0.0013819095', 'U_x': '5.3378769e-12', 'U_y': '6.6061558e-12', 'U_z': '0.19888736', 'p': '0.026034367'}
{'distance': '0.00057286432', 'x': '1.110223e-20', 'y': '1.110223e-20', 'z': '0.0015728643', 'U_x': '6.6005653e-12', 'U_y': '8.0717337e-12', 'U_z': '0.19896816', 'p': '0.02586991'}
{'distance': '0.0007638191', 'x': '1.110223e-20', 'y': '1.110223e-20', 'z': '0.0017638191', 'U_x': '7.8632538e-12', 'U_y': '9.5373116e-12', 'U_z': '0.19904896', 'p': '0.025705453'}

outlet

## Step 3: Pressure Analysis

In OpenFoam : `p` is kinematic pressure.

The centreline sample checks whether the solution has the expected near-linear pressure field. Patch pressure averages provide the boundary pressure drop used for resistance.

In [5]:
for name in ["pressure_summary.csv"]:
    path = OUT_DIR / name
    if path.exists():
        rows = list(csv.DictReader(open(path)))
        for row in rows:
            print(row)
        print(f"Figure: {OUT_DIR / 'plot1_pressure_vs_axial_position.svg'}")
    else:
        print("Pressure data missing; run sampleDict and patchAverage(p).")

{'metric': 'centreline_pressure_gradient_dPdx', 'value': '-711.9553099732326', 'units': 'Pa/m'}
{'metric': 'centreline_linear_fit_R2', 'value': '0.9999911564016399', 'units': '-'}
{'metric': 'centreline_deltaP', 'value': '27.234823577399997', 'units': 'Pa'}
{'metric': 'patch_deltaP', 'value': '28.497418738', 'units': 'Pa'}
Figure: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/output/pipe/analysis/plot1_pressure_vs_axial_position.svg


Pressure drop is a key CFD-to-0D/1D comparison quantity because reduced-order models primarily predict how much pressure is lost to drive a given flow through a segment. Matching local velocity details is useful, but matching the branch pressure-flow relation is what makes a network model physiologically meaningful.

## Step 4: Velocity Analysis

The outlet radial profile is compared with the Poiseuille solution `u(r) = 2 Umean (1 - (r/R)^2)`. The workflow stores RMSE and maximum deviation so that later arterial cases can quantify how far real geometry and boundary conditions move the solution away from ideal laminar pipe flow.

In [6]:
for name in ["velocity_summary.csv"]:
    path = OUT_DIR / name
    if path.exists():
        for row in csv.DictReader(open(path)):
            print(row)
        print(f"Figure: {OUT_DIR / 'plot2_velocity_profile_vs_radius.svg'}")
    else:
        print("Velocity profile missing; run postProcess -func sampleDict -latestTime.")

{'metric': 'Umean_profile_m_per_s', 'value': '0.09945982742602368', 'units': 'm/s'}
{'metric': 'Umax_profile_m_per_s', 'value': '0.19995405', 'units': 'm/s'}
{'metric': 'profile_RMSE_m_per_s', 'value': '0.0005438611354079956', 'units': 'm/s'}
{'metric': 'profile_max_deviation_m_per_s', 'value': '0.001054691000505814', 'units': 'm/s'}
{'metric': 'profile_RMSE_percent_of_Umax', 'value': '0.27199305810909835', 'units': '%'}
Figure: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/output/pipe/analysis/plot2_velocity_profile_vs_radius.svg


## Step 5: Flow Rate Analysis

Flow rate is computed as `Q = Umean * Area` from patch area and patch-average axial velocity. In future 1D models, Q is a primary state variable, and conservation of Q across connected branches is one of the most important checks.

In [7]:
path = OUT_DIR / "flow_rate_summary.csv"
if path.exists():
    rows = list(csv.DictReader(open(path)))
    for row in rows:
        print(row)
    if len(rows) >= 2:
        q0, q1 = abs(float(rows[0]['Q_m3_per_s'])), abs(float(rows[-1]['Q_m3_per_s']))
        print(f"mass-conservation difference = {100*abs(q0-q1)/((q0+q1)/2):.4f}%")
    print(f"Figure: {OUT_DIR / 'plot3_inlet_vs_outlet_flow_rate.svg'}")
else:
    print("Patch-average velocity files are missing.")

{'Location': 'inlet', 'Area_m2': '1.25134469e-05', 'Umean_m_per_s': '0.100780225', 'Q_m3_per_s': '1.2611079941075525e-06'}
{'Location': 'outlet', 'Area_m2': '1.25134469e-05', 'Umean_m_per_s': '0.100780233', 'Q_m3_per_s': '1.2611080942151275e-06'}
mass-conservation difference = 0.0000%
Figure: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/output/pipe/analysis/plot3_inlet_vs_outlet_flow_rate.svg


## Step 6: Hydraulic Resistance Analysis

This is the central section of the workflow. Hydraulic resistance is computed as `R_CFD = DeltaP / Q`. For a straight circular pipe, the analytical comparison is Poiseuille resistance. For real arterial segments, the same CFD extraction remains valid while the reduced-order prediction is built from centreline geometry and local radii.

In [8]:
path = OUT_DIR / "resistance_comparison.csv"
if path.exists():
    for row in csv.DictReader(open(path)):
        print(row)
    print(f"Figure: {OUT_DIR / 'plot4_resistance_comparison.svg'}")
else:
    print("Resistance requires both pressure drop and flow rate.")

{'Quantity': 'DeltaP_Pa', 'CFD': '28.497418738', 'Analytical': '28.218465240000004', 'Error_percent': '0.9885495034101873'}
{'Quantity': 'Q_m3_per_s', 'CFD': '1.2611080942151275e-06', 'Analytical': '1.2664417584794703e-06', 'Error_percent': '-0.4211535373522869'}
{'Quantity': 'R_Pa_s_per_m3', 'CFD': '22597126.18507604', 'Analytical': '22281692.032865357', 'Error_percent': '1.4156651646805782'}
{'Quantity': 'Umax_m_per_s', 'CFD': '0.19995405', 'Analytical': '0.201560466', 'Error_percent': '-0.7969896239473863'}
Figure: /home/gurovamr/OpenFOAM/gurovamr-dev/run/TrackModule2/Hemodynamics-Model-Comparison/output/pipe/analysis/plot4_resistance_comparison.svg


Resistance is the quantity that most directly links CFD, 0D, and 1D models. CFD gives `DeltaP` and `Q` from the resolved field. A 0D model stores that relation as a network element. A 1D model predicts the same relation from vessel geometry, fluid properties, and boundary conditions. This makes resistance the cleanest scalar bridge between high-fidelity and reduced-order hemodynamics.

## Step 7: Quantities Relevant for Future Circle of Willis Simulations

The table below maps quantities across CFD, 1D, and 0D descriptions and marks which are expected to be reused later.

In [9]:
path = OUT_DIR / "future_cow_quantity_map.csv"
for row in csv.DictReader(open(path)):
    print(row)

print("\nGeometry alone gives lengths, radii, areas, and idealized resistance estimates. Pressure, flow rate, mean velocity, peak velocity, and wall shear stress require boundary conditions and a solved flow field. Pressure drop, Q, and resistance will be the main CFD-vs-0D/1D validation quantities; WSS remains a CFD diagnostic and may be compared with approximate 1D estimates only cautiously.")

{'Quantity': 'Pressure', 'CFD': 'field or patch average', '1D': 'nodal/segment pressure', '0D': 'node pressure', 'Used Later?': 'Yes', 'Geometry Alone?': 'No', 'Needs BCs?': 'Yes', 'Validation Use': 'absolute/relative pressure checks'}
{'Quantity': 'Pressure drop', 'CFD': 'inlet-outlet patch pressure', '1D': 'segment pressure loss', '0D': 'R*Q relation', 'Used Later?': 'Yes', 'Geometry Alone?': 'No', 'Needs BCs?': 'Yes', 'Validation Use': 'primary segment comparison'}
{'Quantity': 'Flow rate', 'CFD': 'patch Umean*Area', '1D': 'primary state variable', '0D': 'branch flow', 'Used Later?': 'Yes', 'Geometry Alone?': 'No', 'Needs BCs?': 'Yes', 'Validation Use': 'primary network comparison'}
{'Quantity': 'Resistance', 'CFD': 'DeltaP/Q', '1D': 'geometry/friction closure', '0D': 'network element', 'Used Later?': 'Yes', 'Geometry Alone?': 'Estimate only', 'Needs BCs?': 'CFD yes', 'Validation Use': 'main CFD to 0D/1D bridge'}
{'Quantity': 'Mean velocity', 'CFD': 'patch area average', '1D': 'Q/A'

## Step 8: Transition from Pipe to Real Arterial Segment

The parts that remain unchanged for curved vessels, bifurcations, and Circle of Willis cases are: output discovery, patch-average pressure and velocity loading, pressure-drop calculation, flow-rate calculation, resistance calculation, summary-table generation, and figure export.

The geometry-specific parts will change: the centreline may be curved, profiles may be sampled at multiple outlets, bifurcations will require branch-wise patch maps, and analytical Poiseuille resistance will be replaced by centreline-integrated reduced-order resistance estimates.

Future workflow:

CFD Segment
↓
Extract DeltaP and Q
↓
Compute R_CFD
↓
Extract centreline geometry
↓
Compute R_Poiseuille or 1D resistance
↓
Compare
↓
Validate reduced-order model

This notebook exists to make that sequence reproducible before moving to patient-specific arterial segments and full Circle of Willis simulations.

## Saved Outputs

The notebook writes all analysis artifacts to `output/pipe/analysis`: discovery tables, loaded data exports, pressure/velocity/flow/resistance summaries, future-model mapping tables, and SVG figures.

In [ ]:
for p in sorted(OUT_DIR.iterdir()):
    print(p.name)